In [ ]:
# Cell 1
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# CELL 2 Find the folder that contains train.csv
for dirname, _, filenames in os.walk('/kaggle/input'):
    if 'train.csv' in filenames:
        BASE = dirname
        print("Found it:", BASE)

In [ ]:
# CELL 3 This reads CSVs 

train = pd.read_csv(BASE + '/train.csv')
test = pd.read_csv(BASE + '/test.csv')

print(train.shape, test.shape)
print(train.columns.tolist())
train.head()

In [ ]:
#CELL 4 it will look at lables

print(train['label'].value_counts().sort_index())
print()
print(train['label'].describe())

train['label'].hist(bins=20)


In [ ]:
#CELL 5 Import librosa
import numpy as np, pandas as pd, os
BASE = '/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final'
train = pd.read_csv(BASE + '/train.csv')
test = pd.read_csv(BASE + '/test.csv')



In [ ]:
# Cell 6 

!pip install -q openai-whisper

In [ ]:
# Cell 7
import whisper, torch

print("GPU available:", torch.cuda.is_available())
model = whisper.load_model("small")
print("Model loaded!")

In [ ]:
# Cell 8

import numpy as np
import pandas as pd
import os

In [ ]:
# CELL 9  Base

BASE = '/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final'

train = pd.read_csv(BASE + '/train.csv')
test = pd.read_csv(BASE + '/test.csv')

print(train.shape, test.shape)

In [ ]:
# Cell 10

import whisper, torch

model = whisper.load_model("small")
print("Model loaded!")

In [ ]:
# Cell 11

def transcribe(fname, folder='train'):
    result = model.transcribe(BASE + '/' + folder + '/' + fname, language='en')
    return result['text']

samples = [
    train[train['label'] == 0].iloc[0],
    train[train['label'] == 3].iloc[0],
    train[train['label'] == 5].iloc[0],
]

for row in samples:
    print(row['filename'], "| score:", row['label'])
    print(transcribe(row['filename']))
    print('-' * 60)

In [ ]:
# Cell 12

import json, time
from tqdm.auto import tqdm

OUT = '/kaggle/working/transcripts.csv'

jobs = (
    [(f, 'train') for f in train['filename']] +
    [(f, 'test')  for f in test['filename']]
)

if os.path.exists(OUT):
    done_df = pd.read_csv(OUT)
    rows = done_df.to_dict('records')
    done = set(done_df['filename'])
    print("Resuming. Already done:", len(done))
else:
    rows, done = [], set()

def process(fname, folder):
    result = model.transcribe(
        BASE + '/' + folder + '/' + fname,
        language='en',
        fp16=True,
        verbose=None
    )
    segs = result['segments']
    if len(segs) > 0:
        avg_logprob = sum(s['avg_logprob'] for s in segs) / len(segs)
        no_speech = sum(s['no_speech_prob'] for s in segs) / len(segs)
    else:
        avg_logprob, no_speech = -5.0, 1.0
    return {
        'filename': fname,
        'folder': folder,
        'text': result['text'].strip(),
        'avg_logprob': avg_logprob,
        'no_speech_prob': no_speech,
        'n_segments': len(segs),
    }

start = time.time()
for i, (fname, folder) in enumerate(tqdm(jobs)):
    if fname in done:
        continue
    try:
        rows.append(process(fname, folder))
    except Exception as e:
        print("Failed:", fname, e)
    if len(rows) % 50 == 0:
        pd.DataFrame(rows).to_csv(OUT, index=False)

pd.DataFrame(rows).to_csv(OUT, index=False)
print("Finished! Total:", len(rows), "| minutes:", round((time.time() - start) / 60, 1))

In [ ]:
# Cell 13

t = pd.read_csv(OUT)
print(t.shape)
print(t['folder'].value_counts())
print(t.isna().sum())
t.head()

In [ ]:
# Cell 14

data = t[t['folder'] == 'train'].merge(train, on='filename')
data['text'] = data['text'].fillna('')
data['n_words'] = data['text'].str.split().str.len()

print(data.shape)
print(data.groupby('label')[['n_words', 'avg_logprob', 'no_speech_prob']].mean().round(2))

In [ ]:
# Cell 15

short = data.sort_values('n_words').head(8)
for _, r in short.iterrows():
    print(r['filename'], '| score:', r['label'], '| words:', r['n_words'])
    print(r['text'][:150])
    print('-' * 50)

In [ ]:
# Cell 16

!pip install -q language_tool_python

In [ ]:
# Cell 17

import language_tool_python
tool = language_tool_python.LanguageTool('en-US')

for _, r in data.sample(3, random_state=1).iterrows():
    matches = tool.check(r['text'])
    print('score:', r['label'], '| words:', r['n_words'], '| errors found:', len(matches))
    for m in matches[:3]:
        print('   -', m)

In [ ]:
# Cell 18

print(t.shape)
print(t['folder'].value_counts())

In [ ]:
# Cell 19

data['is_empty'] = (data['n_words'] < 5).astype(int)
print(data.groupby('is_empty')['label'].agg(['count', 'mean']))

In [ ]:
# Cell 20 Use all rows (train + test) from the transcripts table

from tqdm.auto import tqdm

t['text'] = t['text'].fillna('')
t['n_words'] = t['text'].str.split().str.len()

errors = []
for txt in tqdm(t['text']):
    errors.append(len(tool.check(txt)) if len(txt.strip()) > 0 else 0)

t['n_errors'] = errors
t['errors_per_100w'] = t['n_errors'] / t['n_words'].clip(lower=1) * 100

t.to_csv('/kaggle/working/transcripts_with_errors.csv', index=False)
print(t[['n_errors', 'errors_per_100w']].describe().round(2))

In [ ]:
# Cell 21: average error count for each score

check = t[t['folder'] == 'train'].merge(train, on='filename')
print(check.groupby('label')[['n_errors', 'errors_per_100w']].mean().round(2))

In [ ]:

# # Cell 22 Feature table

t['is_empty'] = (t['n_words'] < 5).astype(int)
t['n_sentences'] = t['text'].str.count(r'[.!?]').clip(lower=1)
t['words_per_sentence'] = t['n_words'] / t['n_sentences']
t['unique_ratio'] = t['text'].apply(
    lambda s: len(set(s.lower().split())) / max(len(s.split()), 1)
)

features = ['n_words', 'avg_logprob', 'no_speech_prob', 'n_segments',
            'n_errors', 'errors_per_100w', 'is_empty',
            'n_sentences', 'words_per_sentence', 'unique_ratio']

trainset = t[t['folder'] == 'train'].merge(train, on='filename')
testset = t[t['folder'] == 'test'].copy()

X = trainset[features]
y = trainset['label']
print(X.shape, testset.shape)

In [ ]:
# Cell 23 trainign and testing frst model

from sklearn.model_selection import KFold, cross_val_predict
from sklearn.ensemble import RandomForestRegressor
import numpy as np

model_rf = RandomForestRegressor(n_estimators=300, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

pred = cross_val_predict(model_rf, X, y, cv=kf)
pred = np.clip(pred, 0, 5)

rmse = np.sqrt(((pred - y) ** 2).mean())
corr = np.corrcoef(pred, y)[0, 1]
print("RMSE:", round(rmse, 3), "| Correlation:", round(corr, 3))
print("Lazy baseline RMSE (always guess the average):", round(y.std(), 3))

In [ ]:
# Cell 24  

# Train on ALL 769 training files
model_rf.fit(X, y)

#  Predict the 216 test files
test_pred = np.clip(model_rf.predict(testset[features]), 0, 5)

# Look at the required format
sample = pd.read_csv(BASE + '/sample_submission.csv')
print(sample.head())
print(sample.columns.tolist())

In [ ]:
# Cell 25

sub = pd.DataFrame({'filename': testset['filename'].values, 'label': test_pred})

#  sample_submission.csv, 
sub = sample[['filename']].merge(sub, on='filename', how='left')

print(sub.shape)
print("Missing predictions:", sub['label'].isna().sum())
sub.to_csv('/kaggle/working/submission.csv', index=False)
sub.head()

In [ ]:
# Cell 26

sample_names = set(sample['filename'])
test_csv_names = set(test['filename'])
transcribed_test = set(t[t['folder'] == 'test']['filename'])

import os
test_folder_names = set(os.listdir(BASE + '/test'))

print("sample vs test.csv        :", len(sample_names & test_csv_names), "match of", len(sample_names))
print("sample vs test folder     :", len(sample_names & test_folder_names), "match of", len(sample_names))
print("sample vs transcribed test:", len(sample_names & transcribed_test), "match of", len(sample_names))
print("train vs test overlap     :", len(set(train['filename']) & test_csv_names))

In [ ]:
# Cell 27

pred_df = pd.DataFrame({'filename': testset['filename'].values, 'label': test_pred})

sub = test[['filename']].merge(pred_df, on='filename', how='left')

print(sub.shape)
print("Missing predictions:", sub['label'].isna().sum())
print("Duplicate names:", sub['filename'].duplicated().sum())
sub.to_csv('/kaggle/working/submission.csv', index=False)
sub.head()

In [ ]:
# Cell 28

sample = pd.read_csv(BASE + '/sample_submission.csv')
print("sample rows:", sample.shape[0], "| unique names:", sample['filename'].nunique())

train_folder = set(os.listdir(BASE + '/train'))
test_folder = set(os.listdir(BASE + '/test'))
print("files in train folder:", len(train_folder), "| test folder:", len(test_folder))

print("test.csv names found in test folder :", len(set(test['filename']) & test_folder))
print("sample names found in test folder   :", len(set(sample['filename']) & test_folder))
print("sample names found in train folder  :", len(set(sample['filename']) & train_folder))
print("sample names found in train.csv     :", len(set(sample['filename']) & set(train['filename'])))

In [ ]:
# Cell 29 
print(train.shape, test.shape, len(features), pd.read_csv('/kaggle/working/submission.csv').shape)

In [ ]:
# Cell 30

import os
print(os.listdir('/kaggle/working'))

In [ ]:
# Cell 31

t = pd.read_csv(OUT)
print(t.shape)
print(t.groupby('folder').size())

In [ ]:
# Cell 32

import numpy as np, torch
from tqdm.auto import tqdm

def get_embedding(path):
    audio = whisper.load_audio(path)
    audio = whisper.pad_or_trim(audio)              # 30 seconds
    mel = whisper.log_mel_spectrogram(audio).to(model.device)
    with torch.no_grad():
        enc = model.encoder(mel.unsqueeze(0).half())  # shape: (1, 1500, 768)
    return enc.squeeze(0).float().mean(dim=0).cpu().numpy()

emb_rows = []
for fname, folder in tqdm(jobs):
    emb_rows.append(get_embedding(BASE + '/' + folder + '/' + fname))

emb = np.vstack(emb_rows)
np.save('/kaggle/working/whisper_emb.npy', emb)
print(emb.shape)

In [ ]:
# Cell 33

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_predict

# train rows come first in jobs, so the first 769 embeddings are the training files
n_train = len(train)
X_emb = emb[:n_train]
y_emb = train['label'].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for alpha in [10, 100, 1000, 3000]:
    ridge = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    pred = np.clip(cross_val_predict(ridge, X_emb, y_emb, cv=kf), 0, 5)
    rmse = np.sqrt(((pred - y_emb) ** 2).mean())
    print("alpha", alpha, "| RMSE:", round(rmse, 3))

In [ ]:
# Cell 34

def get_embedding_full(path):
    audio = whisper.load_audio(path)
    chunks = [audio[:480000], audio[480000:960000]]   # two 30-second halves
    vecs = []
    for ch in chunks:
        if len(ch) < 16000:       # skip if second half is almost empty
            continue
        ch = whisper.pad_or_trim(ch)
        mel = whisper.log_mel_spectrogram(ch).to(model.device)
        with torch.no_grad():
            enc = model.encoder(mel.unsqueeze(0).half())
        vecs.append(enc.squeeze(0).float().mean(dim=0).cpu().numpy())
    return np.mean(vecs, axis=0)

emb2 = np.vstack([get_embedding_full(BASE + '/' + folder + '/' + fname)
                  for fname, folder in tqdm(jobs)])
np.save('/kaggle/working/whisper_emb_full.npy', emb2)
print(emb2.shape)

In [ ]:
# Cell 35

X_emb2 = emb2[:n_train]

for alpha in [100, 300, 1000, 3000]:
    ridge = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    pred = np.clip(cross_val_predict(ridge, X_emb2, y_emb, cv=kf), 0, 5)
    rmse = np.sqrt(((pred - y_emb) ** 2).mean())
    print("alpha", alpha, "| RMSE:", round(rmse, 3))

In [ ]:
# Cell 36

ridge = make_pipeline(StandardScaler(), Ridge(alpha=300))
ridge.fit(X_emb2, y_emb)

test_emb2 = emb2[n_train:]                      # test rows come after train in jobs
test_pred2 = np.clip(ridge.predict(test_emb2), 0, 5)

# jobs order for test = order of test.csv
sub2 = pd.DataFrame({'filename': test['filename'].values, 'label': test_pred2})
print(sub2.shape, sub2['label'].isna().sum())
sub2.to_csv('/kaggle/working/submission.csv', index=False)
sub2.head()

In [ ]:
# Cell 37

chk = pd.read_csv('/kaggle/working/submission.csv')
print(chk.shape)
print(chk['label'].describe().round(3))
print(chk.head(3))

In [ ]:
# Cell 38

def get_layer_embs(path, layers=(4, 8, 12)):
    audio = whisper.load_audio(path)
    chunks = [audio[:480000], audio[480000:960000]]
    store = {}
    hooks = []
    for i in layers:
        blk = model.encoder.blocks[i - 1]
        hooks.append(blk.register_forward_hook(
            lambda m, inp, out, i=i: store.__setitem__(i, out)))
    per_layer = {i: [] for i in layers}
    for ch in chunks:
        if len(ch) < 16000:
            continue
        ch = whisper.pad_or_trim(ch)
        mel = whisper.log_mel_spectrogram(ch).to(model.device)
        with torch.no_grad():
            model.encoder(mel.unsqueeze(0).half())
        for i in layers:
            per_layer[i].append(store[i].squeeze(0).float().mean(dim=0).cpu().numpy())
    for h in hooks:
        h.remove()
    return {i: np.mean(per_layer[i], axis=0) for i in layers}

layer_rows = [get_layer_embs(BASE + '/' + f + '/' + n) if False else
              get_layer_embs(BASE + '/' + folder + '/' + fname)
              for fname, folder in tqdm(jobs)]

layer_embs = {i: np.vstack([r[i] for r in layer_rows]) for i in (4, 8, 12)}
for i, e in layer_embs.items():
    np.save(f'/kaggle/working/whisper_layer{i}.npy', e)
print({i: e.shape for i, e in layer_embs.items()})

In [ ]:
# Cell 39

for i, e in layer_embs.items():
    X_l = e[:n_train]
    for alpha in [100, 300, 1000]:
        ridge = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
        pred = np.clip(cross_val_predict(ridge, X_l, y_emb, cv=kf), 0, 5)
        rmse = np.sqrt(((pred - y_emb) ** 2).mean())
        print("layer", i, "| alpha", alpha, "| RMSE:", round(rmse, 3))

In [ ]:
# Cell 40

combos = {
    '8+12': np.hstack([layer_embs[8], layer_embs[12]]),
    '4+8+12': np.hstack([layer_embs[4], layer_embs[8], layer_embs[12]]),
    '8+12+final': np.hstack([layer_embs[8], layer_embs[12], emb2]),
}

for name, e in combos.items():
    X_c = e[:n_train]
    for alpha in [300, 1000, 3000]:
        ridge = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
        pred = np.clip(cross_val_predict(ridge, X_c, y_emb, cv=kf), 0, 5)
        rmse = np.sqrt(((pred - y_emb) ** 2).mean())
        print(name, "| alpha", alpha, "| RMSE:", round(rmse, 3))

In [ ]:
# Cell 41

X_best = np.hstack([layer_embs[4], layer_embs[8], layer_embs[12]])
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1000))
ridge.fit(X_best[:n_train], y_emb)

test_pred3 = np.clip(ridge.predict(X_best[n_train:]), 0, 5)
sub3 = pd.DataFrame({'filename': test['filename'].values, 'label': test_pred3})
print(sub3.shape, sub3['label'].isna().sum())
sub3.to_csv('/kaggle/working/submission.csv', index=False)
sub3.head(3)

In [ ]:
# Cell 42 Text Embeddings

!pip install -q sentence-transformers

In [ ]:
# Cell 42

from sentence_transformers import SentenceTransformer

t = pd.read_csv('/kaggle/working/transcripts.csv')
t['text'] = t['text'].fillna('')

# align to jobs order (train first, then test)
order = pd.DataFrame(jobs, columns=['filename', 'folder'])
t = order.merge(t, on=['filename', 'folder'], how='left')
print(t.shape, t['text'].isna().sum())

st = SentenceTransformer('all-mpnet-base-v2')
txt_emb = st.encode(t['text'].fillna('').tolist(), show_progress_bar=True, batch_size=32)
np.save('/kaggle/working/text_emb.npy', txt_emb)
print(txt_emb.shape)

In [ ]:
# Cell 45

X_audio = np.hstack([layer_embs[4], layer_embs[8], layer_embs[12]])
X_text = txt_emb
X_both = np.hstack([X_audio, X_text])

for name, e in {'text only': X_text, 'audio only': X_audio, 'audio+text': X_both}.items():
    for alpha in [300, 1000, 3000]:
        ridge = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
        pred = np.clip(cross_val_predict(ridge, e[:n_train], y_emb, cv=kf), 0, 5)
        rmse = np.sqrt(((pred - y_emb) ** 2).mean())
        print(name, "| alpha", alpha, "| RMSE:", round(rmse, 3))

In [ ]:
# Cell 46

!pip install -q transformers


In [ ]:
# Cell 47

from transformers import AutoFeatureExtractor, WavLMModel
import librosa, torch, numpy as np
from tqdm.auto import tqdm

fe = AutoFeatureExtractor.from_pretrained('microsoft/wavlm-base-plus')
wavlm = WavLMModel.from_pretrained('microsoft/wavlm-base-plus').to('cuda').eval()

def wavlm_embed(path, layers=(4, 8, 12)):
    audio, _ = librosa.load(path, sr=16000)
    audio = audio[:16000 * 60]                       # at most 60 seconds
    inputs = fe(audio, sampling_rate=16000, return_tensors='pt')
    with torch.no_grad():
        out = wavlm(inputs['input_values'].to('cuda'), output_hidden_states=True)
    return np.concatenate(
        [out.hidden_states[i].squeeze(0).mean(dim=0).cpu().numpy() for i in layers]
    )

wav_emb = np.vstack([wavlm_embed(BASE + '/' + folder + '/' + fname)
                     for fname, folder in tqdm(jobs)])
np.save('/kaggle/working/wavlm_emb.npy', wav_emb)
print(wav_emb.shape)

In [ ]:
# Cell 48

for name, e in {'wavlm': wav_emb, 'whisper+wavlm': np.hstack([X_audio, wav_emb])}.items():
    for alpha in [1000, 3000, 10000]:
        ridge = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
        pred = np.clip(cross_val_predict(ridge, e[:n_train], y_emb, cv=kf), 0, 5)
        rmse = np.sqrt(((pred - y_emb) ** 2).mean())
        print(name, "| alpha", alpha, "| RMSE:", round(rmse, 3))

In [ ]:
# Cell 49 

def wavlm_embed_ms(path, layers=(4, 8, 12)):
    audio, _ = librosa.load(path, sr=16000)
    audio = audio[:16000 * 60]
    inputs = fe(audio, sampling_rate=16000, return_tensors='pt')
    with torch.no_grad():
        out = wavlm(inputs['input_values'].to('cuda'), output_hidden_states=True)
    parts = []
    for i in layers:
        h = out.hidden_states[i].squeeze(0)
        parts += [h.mean(dim=0).cpu().numpy(), h.std(dim=0).cpu().numpy()]
    return np.concatenate(parts)

wav_ms = np.vstack([wavlm_embed_ms(BASE + '/' + folder + '/' + fname)
                    for fname, folder in tqdm(jobs)])
np.save('/kaggle/working/wavlm_ms.npy', wav_ms)
print(wav_ms.shape)

In [ ]:
# Cell 50

for name, e in {'wavlm mean+std': wav_ms,
                'whisper+wavlm mean+std': np.hstack([X_audio, wav_ms])}.items():
    for alpha in [1000, 3000, 10000]:
        ridge = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
        pred = np.clip(cross_val_predict(ridge, e[:n_train], y_emb, cv=kf), 0, 5)
        rmse = np.sqrt(((pred - y_emb) ** 2).mean())
        print(name, "| alpha", alpha, "| RMSE:", round(rmse, 3))

In [ ]:
# Cell 51

from sklearn.svm import SVR
from sklearn.decomposition import PCA

X_best2 = np.hstack([X_audio, wav_ms])

for C in [1, 3, 10]:
    svr = make_pipeline(StandardScaler(), PCA(n_components=200, random_state=42), SVR(C=C, kernel='rbf'))
    pred = np.clip(cross_val_predict(svr, X_best2[:n_train], y_emb, cv=kf), 0, 5)
    rmse = np.sqrt(((pred - y_emb) ** 2).mean())
    print("SVR C", C, "| RMSE:", round(rmse, 3))

In [ ]:
import os
print(sorted(os.listdir('/kaggle/working')))

In [ ]:
import os
print(sorted(os.listdir('/kaggle/working')))

In [ ]:
print("alive")